In [24]:
# ================================================================
# 🚨 MANEJO DE ERRORES PIPELINE - SILVER
# ================================================================
import traceback
from datetime import datetime
from zoneinfo import ZoneInfo
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp, date_format

CATALOG = "Prueba Tecnica Dataknow"
GOLD_LH = "LH_GOLD_PRUEBA"
SCHEMA = "dbo"

PIPELINE_NAME = "GOLD"
BOGOTA_TZ = ZoneInfo("America/Bogota")

pipeline_errors = []

def ejecutar_con_manejo_errores(nombre_tarea, funcion):
    try:
        funcion()

        pipeline_errors.append(Row(
            pipeline=PIPELINE_NAME,
            tarea=nombre_tarea,
            estado="OK",
            mensaje="Ejecutado correctamente",
            detalle_error="",
            timestamp=datetime.now(BOGOTA_TZ).strftime("%Y-%m-%d %H:%M:%S")
        ))

        logger.info(f"✅ {nombre_tarea}")

    except Exception as e:
        pipeline_errors.append(Row(
            pipeline=PIPELINE_NAME,
            tarea=nombre_tarea,
            estado="ERROR",
            mensaje=str(e)[:500],
            detalle_error=traceback.format_exc()[:2000],
            timestamp=datetime.now(BOGOTA_TZ).strftime("%Y-%m-%d %H:%M:%S")
        ))

        logger.error(f"⚠️ Error en {nombre_tarea}: {str(e)[:200]}")
        logger.error("→ Registrado. Continuando...")

def guardar_errores_pipeline():
    if pipeline_errors:
        df = spark.createDataFrame(pipeline_errors)

        df = df.withColumn(
            "_fec_registro",
            date_format(current_timestamp(), "yyyy-MM-dd HH:mm:ss")
        )

        df.write.format("delta") \
          .mode("append") \
          .option("mergeSchema", "true") \
          .saveAsTable(f"`{CATALOG}`.{GOLD_LH}.{SCHEMA}.pipeline_error_log")
        ok = len([e for e in pipeline_errors if e.estado == "OK"])
        err = len([e for e in pipeline_errors if e.estado == "ERROR"])

        logger.info(f"\n📋 pipeline_error_log BRONZE: {ok} OK, {err} errores")

StatementMeta(, 575c5f61-7051-4b2e-8c76-6e33f2202ba1, 26, Finished, Available, Finished, False)

In [25]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  GOLD - MODELO ANALÍTICO Y REGLAS DE NEGOCIO                       ║
# ║  Lee: SLV_* (Silver)                                                ║
# ║  Crea: dim_* (dimensiones) + fact_* (hechos) + kpi_*               ║
# ╚══════════════════════════════════════════════════════════════════════╝

from pyspark.sql.functions import (
    col, lit, when, concat_ws, floor, datediff, current_date,
    months_between, round as spark_round, avg, sum as spark_sum,
    count, countDistinct, max as spark_max, min as spark_min,
    month, year, date_format, expr, pow as spark_pow,
    lag, stddev, coalesce, upper, trim, to_date
)
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, IntegerType, StringType
from pyspark.sql.functions import dayofweek
import logging
import time

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("GoldPipeline")

# ---------------------------
# Configuración
# ---------------------------
CATALOG = "Prueba Tecnica Dataknow"
SILVER_LH = "LH_SILVER_PRUEBA"
GOLD_LH = "LH_GOLD_PRUEBA"          
SCHEMA = "dbo"

TRM_COP_USD = 4150.0                 # Tasa de cambio COP → USD

def slv(t):
    return f"`{CATALOG}`.{SILVER_LH}.{SCHEMA}.slv_{t}"

def gold(t):
    return f"`{CATALOG}`.{GOLD_LH}.{SCHEMA}.{t}"

StatementMeta(, 575c5f61-7051-4b2e-8c76-6e33f2202ba1, 27, Finished, Available, Finished, False)

In [26]:

# ================================================================
# DIM_CLIENTES
# Origen: slv_tb_clientes_core
# Transformaciones:
#   - Unificar nombre completo (nomb_cli + apell_cli)
#   - Calcular edad desde fec_nac
#   - Mapear cod_segmento a etiqueta legible
# ================================================================
def crear_dim_clientes():
    logger.info(f"\n{'=' * 60}")
    logger.info("📋 dim_clientes")
    logger.info(f"{'=' * 60}")

    df = spark.sql(f"SELECT * FROM {slv('tb_clientes_core')}")

    df_dim = df.select(
        col("id_cli"),
        concat_ws(" ", col("nomb_cli"), col("apell_cli")).alias("nombre_completo"),
        col("nomb_cli"),
        col("apell_cli"),
        col("tip_doc"),
        col("num_doc_hash"),
        col("fec_nac"),
        col("fec_alta"),

        # Edad calculada
        floor(datediff(current_date(), col("fec_nac")) / 365.25).cast(IntegerType()).alias("edad"),

        # Rango de edad
        when(floor(datediff(current_date(), col("fec_nac")) / 365.25) < 25, "18-24")
        .when(floor(datediff(current_date(), col("fec_nac")) / 365.25) < 35, "25-34")
        .when(floor(datediff(current_date(), col("fec_nac")) / 365.25) < 45, "35-44")
        .when(floor(datediff(current_date(), col("fec_nac")) / 365.25) < 55, "45-54")
        .when(floor(datediff(current_date(), col("fec_nac")) / 365.25) < 65, "55-64")
        .otherwise("65+").alias("rango_edad"),

        # mapeo con los códigos reales
        when(col("cod_segmento") == "B", "BASICO")
        .when(col("cod_segmento") == "E", "ESTANDAR")
        .when(col("cod_segmento") == "P", "PREMIUM")
        .when(col("cod_segmento") == "S", "ELITE")
        .otherwise("SIN_SEGMENTO").alias("segmento"),


        col("cod_segmento"),
        col("score_buro"),
        col("ciudad_res"),
        col("depto_res"),
        col("estado_cli"),
        col("canal_adquis"),

        # Antigüedad en meses
        floor(months_between(current_date(), col("fec_alta"))).cast(IntegerType()).alias("antiguedad_meses"),
    )

    df_dim.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(gold("dim_clientes"))

    cnt = df_dim.count()
    logger.info(f"  ✅ {cnt:,} registros → {gold('dim_clientes')}")
    return df_dim

StatementMeta(, 575c5f61-7051-4b2e-8c76-6e33f2202ba1, 28, Finished, Available, Finished, False)

In [27]:
# ================================================================
# DIM_PRODUCTOS
# Origen: slv_tb_productos_cat
# Transformaciones:
#   - Renombrar campos a nombres de negocio
#   - Calcular tasa mensual equivalente desde tasa_ea
#   - Clasificar en familia: crédito, ahorro, transaccional
# ================================================================
def crear_dim_productos():
    logger.info(f"\n{'=' * 60}")
    logger.info("📋 dim_productos")
    logger.info(f"{'=' * 60}")

    df = spark.sql(f"SELECT * FROM {slv('tb_productos_cat')}")

    df_dim = df.select(
        col("cod_prod").alias("codigo_producto"),
        col("desc_prod").alias("nombre_producto"),
        col("tip_prod").alias("tipo_producto"),

        # Familia de producto
        when(col("tip_prod").isin("CREDITO_CONSUMO", "CREDITO_ROTATIVO", "TARJETA_DIGITAL", "CREDITO"), "CREDITO")
        .when(col("tip_prod").isin("CUENTA_AHORRO", "AHORRO", "AHORRO_DIGITAL"), "AHORRO")
        .otherwise("TRANSACCIONAL").alias("familia_producto"),

        col("tasa_ea"),

        # Tasa mensual equivalente: (1 + tasa_ea)^(1/12) - 1
        spark_round(
            (spark_pow(lit(1) + col("tasa_ea") / lit(100), lit(1.0 / 12)) - lit(1)) * lit(100),
            4
        ).alias("tasa_mensual_equiv"),

        col("plazo_max_meses"),
        col("cuota_min").alias("cuota_minima"),
        col("comision_admin").alias("comision_administracion"),
        col("estado_prod").alias("estado_producto"),
    )

    df_dim.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(gold("dim_productos"))

    cnt = df_dim.count()
    logger.info(f"  ✅ {cnt:,} registros → {gold('dim_productos')}")
    return df_dim

StatementMeta(, 575c5f61-7051-4b2e-8c76-6e33f2202ba1, 29, Finished, Available, Finished, False)

In [28]:
# ================================================================
# DIM_GEOGRAFIA
# Origen: slv_tb_sucursales_red
# Transformaciones:
#   - Separar en dimensión geográfica con ciudad y depto
# ================================================================
def crear_dim_geografia():
    logger.info(f"\n{'=' * 60}")
    logger.info("📋 dim_geografia")
    logger.info(f"{'=' * 60}")

    df = spark.sql(f"SELECT * FROM {slv('tb_sucursales_red')}")

    df_dim = df.select(
        col("cod_suc").alias("codigo_sucursal"),
        col("nom_suc").alias("nombre_sucursal"),
        col("ciudad"),
        col("depto").alias("departamento"),
        col("latitud"),
        col("longitud"),
        col("activo"),
    ).dropDuplicates(["ciudad", "departamento", "codigo_sucursal"])

    df_dim.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(gold("dim_geografia"))

    cnt = df_dim.count()
    logger.info(f"  ✅ {cnt:,} registros → {gold('dim_geografia')}")
    return df_dim



StatementMeta(, 575c5f61-7051-4b2e-8c76-6e33f2202ba1, 30, Finished, Available, Finished, False)

In [29]:
# ================================================================
# DIM_CANAL
# Origen: slv_tb_sucursales_red
# Transformaciones:
#   - Tipo de punto de atención y canal digital
# ================================================================
def crear_dim_canal():
    logger.info(f"\n{'=' * 60}")
    logger.info("📋 dim_canal")
    logger.info(f"{'=' * 60}")

    df = spark.sql(f"SELECT * FROM {slv('tb_sucursales_red')}")

    df_dim = df.select(
        col("cod_suc").alias("codigo_sucursal"),
        col("tip_punto").alias("tipo_punto_atencion"),

        # Clasificar canal
        when(col("tip_punto").isin("APP", "WEB", "PORTAL"), "DIGITAL")
        .when(col("tip_punto").isin("CORRESPONSAL", "CORRESPONSAL_BANCARIO"), "CORRESPONSAL")
        .when(col("tip_punto").isin("OFICINA", "SUCURSAL"), "PRESENCIAL")
        .otherwise("OTRO").alias("canal"),

        col("activo"),
    ).dropDuplicates(["codigo_sucursal"])

    df_dim.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(gold("dim_canal"))

    cnt = df_dim.count()
    logger.info(f"  ✅ {cnt:,} registros → {gold('dim_canal')}")
    return df_dim


StatementMeta(, 575c5f61-7051-4b2e-8c76-6e33f2202ba1, 31, Finished, Available, Finished, False)

In [30]:
# ================================================================
# FACT_TRANSACCIONES
# Origen: slv_tb_mov_financieros
# Transformaciones:
#   - Validar id_cli contra dim_clientes
#   - Calcular monto en USD desde COP
#   - Flag de horario (hábil / no hábil)
#   - Promedio móvil 30 días por cliente
#   - ind_sospechoso (vr_mov > promedio + 3*stddev)
# ================================================================
def crear_fact_transacciones():
    logger.info(f"\n{'=' * 60}")
    logger.info("📋 fact_transacciones")
    logger.info(f"{'=' * 60}")

    df = spark.sql(f"SELECT * FROM {slv('tb_mov_financieros')}")

    # Extraer hora como entero
    df = df.withColumn("hora", col("hra_mov").substr(1, 2).cast(IntegerType()))

    # Flag horario (dayofweek: 2=Lunes a 6=Viernes)
    df = df.withColumn(
        "flag_horario",
        when(
            (dayofweek(col("fec_mov")).between(2, 6)) &
            (col("hora").between(6, 18)),
            lit("HABIL")
        ).otherwise(lit("NO_HABIL"))
    )

    # Monto en USD
    df = df.withColumn(
        "vr_mov_usd",
        spark_round(col("vr_mov") / lit(TRM_COP_USD), 2)
    )

    df = df.filter(col("fec_mov").isNotNull())
    # Seleccionar columnas — ind_sospechoso, prom_movil_30d, stddev_movil_30d
    df_fact = df.select(
        col("id_mov"),
        col("id_cli"),
        col("cod_prod"),
        col("num_cuenta"),
        col("fec_mov"),
        col("hra_mov"),
        col("hora"),
        col("vr_mov"),
        col("vr_mov_usd"),
        col("tip_mov"),
        col("cod_canal"),
        col("cod_ciudad"),
        col("cod_estado_mov"),
        col("id_dispositivo"),
        col("flag_horario"),
        col("prom_movil_30d"),       #de Silver
        col("stddev_movil_30d"),     #de Silver
        col("ind_sospechoso"),       #de Silver
        year(col("fec_mov")).alias("anio"),
        month(col("fec_mov")).alias("mes"),
    )

    df_fact.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true") \
        .partitionBy("anio", "mes") \
        .saveAsTable(gold("fact_transacciones"))

    cnt = df_fact.count()
    sosp = df_fact.filter(col("ind_sospechoso") == True).count()
    logger.info(f"  ✅ {cnt:,} registros → {gold('fact_transacciones')}")
    logger.info(f"  🔴 {sosp:,} transacciones sospechosas detectadas")
    return df_fact

StatementMeta(, 575c5f61-7051-4b2e-8c76-6e33f2202ba1, 32, Finished, Available, Finished, False)

In [31]:
# ================================================================
# FACT_CARTERA
# Origen: slv_tb_obligaciones
# Transformaciones:
#   - bucket_mora: Al día, Rango 1-4, Deteriorado
#   - Clasificación regulatoria A/B/C/D/E
#   - Provisión estimada según tabla regulatoria
# ================================================================
def crear_fact_cartera():
    logger.info(f"\n{'=' * 60}")
    logger.info("📋 fact_cartera")
    logger.info(f"{'=' * 60}")

    df = spark.sql(f"SELECT * FROM {slv('tb_obligaciones')}")

    df_fact = df.select(
        col("id_oblig"),
        col("id_cli"),
        col("cod_prod"),
        col("vr_aprobado"),
        col("vr_desembolsado"),
        col("sdo_capital"),
        col("vr_cuota"),
        col("fec_desembolso"),
        col("fec_venc"),
        col("dias_mora_act"),
        col("num_cuotas_pend"),
        col("calif_riesgo"),

        # bucket_mora: 5 rangos
        when(col("dias_mora_act") == 0, "AL_DIA")
        .when(col("dias_mora_act").between(1, 30), "RANGO_1")
        .when(col("dias_mora_act").between(31, 60), "RANGO_2")
        .when(col("dias_mora_act").between(61, 90), "RANGO_3")
        .otherwise("DETERIORADO").alias("bucket_mora"),

        # Clasificación regulatoria
        when(col("dias_mora_act") == 0, "A")
        .when(col("dias_mora_act").between(1, 30), "B")
        .when(col("dias_mora_act").between(31, 60), "C")
        .when(col("dias_mora_act").between(61, 90), "D")
        .otherwise("E").alias("clasif_regulatoria"),

        # Provisión estimada (% del saldo según regulación SFC)
        spark_round(
            col("sdo_capital") *
            when(col("dias_mora_act") == 0, lit(0.01))         # A: 1%
            .when(col("dias_mora_act").between(1, 30), lit(0.032))  # B: 3.2%
            .when(col("dias_mora_act").between(31, 60), lit(0.10))  # C: 10%
            .when(col("dias_mora_act").between(61, 90), lit(0.20))  # D: 20%
            .otherwise(lit(0.50)),                                   # E: 50%
            2
        ).alias("provision_estimada"),

        # Porcentaje de avance de pago
        spark_round(
            (lit(1) - col("sdo_capital") / col("vr_desembolsado")) * lit(100),
            2
        ).alias("pct_avance_pago"),
    )

    df_fact.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(gold("fact_cartera"))

    cnt = df_fact.count()
    det = df_fact.filter(col("bucket_mora") == "DETERIORADO").count()
    logger.info(f"  ✅ {cnt:,} registros → {gold('fact_cartera')}")
    logger.info(f"  🔴 {det:,} obligaciones deterioradas (>90 días)")
    return df_fact



StatementMeta(, 575c5f61-7051-4b2e-8c76-6e33f2202ba1, 33, Finished, Available, Finished, False)

In [32]:
# ================================================================
# FACT_RENTABILIDAD_CLIENTE
# Origen: slv_tb_comisiones_log + slv_tb_obligaciones
# Transformaciones:
#   - Join por id_cli y periodo mensual
#   - Ingreso total = intereses + comisiones
#   - CLTV = suma histórica 12 meses
# ================================================================
def crear_fact_rentabilidad():
    logger.info(f"\n{'=' * 60}")
    logger.info("📋 fact_rentabilidad_cliente")
    logger.info(f"{'=' * 60}")

    # Comisiones cobradas por cliente/mes
    df_com = spark.sql(f"SELECT * FROM {slv('tb_comisiones_log')}")
    df_com = df_com.filter(upper(col("estado_cobro")) == "COBRADO")

    df_com_mes = df_com.groupBy(
        col("id_cli"),
        year(col("fec_cobro")).alias("anio"),
        month(col("fec_cobro")).alias("mes"),
    ).agg(
        spark_sum("vr_comision").alias("ingresos_comisiones"),
        count("id_comision").alias("num_comisiones"),
    )

    # Intereses por cliente/mes (estimado: sdo_capital * tasa_ea/12)
    df_obl = spark.sql(f"SELECT * FROM {slv('tb_obligaciones')}")
    df_prod = spark.sql(f"SELECT cod_prod, tasa_ea FROM {slv('tb_productos_cat')}")

    df_obl_prod = df_obl.join(df_prod, "cod_prod", "left")

    df_int_mes = df_obl_prod.filter(
        col("fec_desembolso").isNotNull()
    ).groupBy(
        col("id_cli"),
        year(col("fec_desembolso")).alias("anio"),
        month(col("fec_desembolso")).alias("mes"),
    ).agg(
        spark_round(
            spark_sum(col("sdo_capital") * col("tasa_ea") / lit(12)),
            2
        ).alias("ingresos_intereses"),
        spark_sum("sdo_capital").alias("saldo_total"),
        count("id_oblig").alias("num_obligaciones"),
    )

    df_rent = df_com_mes.join(
        df_int_mes,
        ["id_cli", "anio", "mes"],
        "full_outer"
    )

    df_rent = df_rent.withColumn(
        "ingresos_comisiones", coalesce(col("ingresos_comisiones"), lit(0))
    ).withColumn(
        "ingresos_intereses", coalesce(col("ingresos_intereses"), lit(0))
    ).withColumn(
        "ingreso_total",
        spark_round(col("ingresos_comisiones") + col("ingresos_intereses"), 2)
    )

    # Filtrar registros sin anio/mes 
    df_rent = df_rent.filter(col("anio").isNotNull() & col("mes").isNotNull())

    w_12m = Window.partitionBy("id_cli") \
        .orderBy("anio", "mes") \
        .rowsBetween(-11, 0)

    df_rent = df_rent.withColumn(
        "cltv_12m",
        spark_round(spark_sum("ingreso_total").over(w_12m), 2)
    )

    df_fact = df_rent.select(
        col("id_cli"),
        col("anio"),
        col("mes"),
        col("ingresos_comisiones"),
        col("ingresos_intereses"),
        col("ingreso_total"),
        coalesce(col("num_comisiones"), lit(0)).alias("num_comisiones"),
        coalesce(col("num_obligaciones"), lit(0)).alias("num_obligaciones"),
        coalesce(col("saldo_total"), lit(0)).alias("saldo_total"),
        col("cltv_12m"),
    )

    df_fact.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true") \
        .partitionBy("anio") \
        .saveAsTable(gold("fact_rentabilidad_cliente"))

    cnt = df_fact.count()
    logger.info(f"  ✅ {cnt:,} registros → {gold('fact_rentabilidad_cliente')}")
    return df_fact



StatementMeta(, 575c5f61-7051-4b2e-8c76-6e33f2202ba1, 34, Finished, Available, Finished, False)

In [33]:
# ================================================================
# KPI_DIARIO_CARTERA
# Agregar por fecha, producto, segmento y ciudad:
#   - Total obligaciones activas
#   - Monto total cartera
#   - Monto en mora
#   - Tasa de mora (%)
#   - Clientes con alguna obligación en mora
# ================================================================
def crear_kpi_diario_cartera():
    logger.info(f"\n{'=' * 60}")
    logger.info("📋 kpi_diario_cartera")
    logger.info(f"{'=' * 60}")

    df_obl = spark.sql(f"SELECT * FROM {slv('tb_obligaciones')}")
    df_cli = spark.sql(f"""
        SELECT id_cli, cod_segmento, ciudad_res 
        FROM {slv('tb_clientes_core')}
    """)
    df_prod = spark.sql(f"""
        SELECT cod_prod, desc_prod, tip_prod 
        FROM {slv('tb_productos_cat')}
    """)

    df = df_obl.join(df_cli, "id_cli", "left") \
            .join(df_prod, "cod_prod", "left")

    # Segmento legible
    df = df.withColumn(
        "segmento",
        when(col("cod_segmento") == "B", "BASICO")
        .when(col("cod_segmento") == "E", "ESTANDAR")
        .when(col("cod_segmento") == "P", "PREMIUM")
        .when(col("cod_segmento") == "S", "ELITE")
        .otherwise("SIN_SEGMENTO")
    )

    df = df.withColumn("fecha", col("fec_desembolso"))

    # Agrupar por: fecha, producto (cod_prod), segmento, ciudad
    df_kpi = df.filter(col("fecha").isNotNull()).groupBy(
        col("fecha"),
        col("cod_prod"),
        col("desc_prod").alias("nombre_producto"),
        col("tip_prod").alias("tipo_producto"),
        col("segmento"),
        col("ciudad_res").alias("ciudad"),
    ).agg(
        # Total obligaciones activas
        count("id_oblig").alias("total_obligaciones"),

        # Monto total cartera
        spark_round(spark_sum("sdo_capital"), 2).alias("monto_cartera"),

        # Monto en mora
        spark_round(
            spark_sum(when(col("dias_mora_act") > 0, col("sdo_capital")).otherwise(0)),
            2
        ).alias("monto_mora"),

        # Clientes con alguna obligación en mora
        countDistinct(
            when(col("dias_mora_act") > 0, col("id_cli"))
        ).alias("clientes_en_mora"),

        # Total clientes
        countDistinct("id_cli").alias("total_clientes"),

        # Provisión total
        spark_round(
            spark_sum(
                col("sdo_capital") *
                when(col("dias_mora_act") == 0, lit(0.01))
                .when(col("dias_mora_act").between(1, 30), lit(0.032))
                .when(col("dias_mora_act").between(31, 60), lit(0.10))
                .when(col("dias_mora_act").between(61, 90), lit(0.20))
                .otherwise(lit(0.50))
            ), 2
        ).alias("provision_total"),
    )

    # Tasa de mora (%)
    df_kpi = df_kpi.withColumn(
        "tasa_mora_pct",
        spark_round(
            when(col("monto_cartera") > 0,
                col("monto_mora") / col("monto_cartera") * lit(100)
            ).otherwise(lit(0)),
            2
        )
    )

    df_kpi.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true") \
        .partitionBy("segmento") \
        .saveAsTable(gold("kpi_diario_cartera"))

    cnt = df_kpi.count()
    logger.info(f"  ✅ {cnt:,} registros → {gold('kpi_diario_cartera')}")
    return df_kpi



StatementMeta(, 575c5f61-7051-4b2e-8c76-6e33f2202ba1, 35, Finished, Available, Finished, False)

In [34]:
# ================================================================
# LINAJE DE DATOS (3 campos calculados documentados)
# ================================================================
def documentar_linaje():
    from pyspark.sql import Row

    linaje = [
        Row(
            campo_calculado="fact_cartera.bucket_mora",
            tabla_origen="slv_tb_obligaciones",
            campo_origen="dias_mora_act",
            transformacion="CASE: 0='AL_DIA', 1-30='RANGO_1', 31-60='RANGO_2', 61-90='RANGO_3', >90='DETERIORADO'",
            proposito="Clasificar obligaciones en rangos de mora para monitoreo de riesgo crediticio"
        ),
        Row(
            campo_calculado="fact_transacciones.ind_sospechoso",
            tabla_origen="slv_tb_mov_financieros",
            campo_origen="vr_mov",
            transformacion="True cuando vr_mov > promedio_movil_30d + 3 * desviacion_estandar_30d del mismo cliente",
            proposito="Detectar transacciones atípicas para alimentar el motor de prevención de fraude"
        ),
        Row(
            campo_calculado="fact_rentabilidad_cliente.cltv_12m",
            tabla_origen="slv_tb_comisiones_log + slv_tb_obligaciones + slv_tb_productos_cat",
            campo_origen="vr_comision + sdo_capital * tasa_ea",
            transformacion="Suma rolling 12 meses de (ingresos_comisiones_cobradas + ingresos_intereses_estimados) por cliente",
            proposito="Customer Lifetime Value para segmentación comercial y decisiones de oferta"
        ),
        Row(
            campo_calculado="fact_cartera.provision_estimada",
            tabla_origen="slv_tb_obligaciones",
            campo_origen="sdo_capital + dias_mora_act",
            transformacion="sdo_capital * porcentaje_regulatorio (A=1%, B=3.2%, C=10%, D=20%, E=50%)",
            proposito="Estimación de provisiones regulatorias exigidas por la Superintendencia Financiera"
        ),
        Row(
            campo_calculado="dim_clientes.edad",
            tabla_origen="slv_tb_clientes_core",
            campo_origen="fec_nac",
            transformacion="FLOOR(DATEDIFF(current_date, fec_nac) / 365.25)",
            proposito="Edad del cliente para segmentación demográfica y análisis de riesgo por edad"
        ),
        Row(
            campo_calculado="dim_productos.tasa_mensual_equiv",
            tabla_origen="slv_tb_productos_cat",
            campo_origen="tasa_ea",
            transformacion="((1 + tasa_ea/100)^(1/12) - 1) * 100",
            proposito="Tasa mensual equivalente para cálculo de intereses mensuales y comparación entre productos"
        ),
    ]

    df_linaje = spark.createDataFrame(linaje)
    df_linaje.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(gold("gold_linaje_datos"))

    logger.info(f"\n  📋 Linaje documentado: {gold('gold_linaje_datos')} ({len(linaje)} campos)")

# ================================================================
# EJECUCIÓN
# ================================================================
if __name__ == "__main__":
    t0 = time.time()

    logger.info(f"\n{'=' * 60}")
    logger.info("🥇 PIPELINE GOLD — MODELO ANALÍTICO")
    logger.info(f"   Origen:  {SILVER_LH}")
    logger.info(f"   Destino: {GOLD_LH}")
    logger.info(f"{'=' * 60}")

    # ============================================================
    # DIMENSIONES
    # ============================================================
    ejecutar_con_manejo_errores("DIM_CLIENTES", crear_dim_clientes)
    ejecutar_con_manejo_errores("DIM_PRODUCTOS", crear_dim_productos)
    ejecutar_con_manejo_errores("DIM_GEOGRAFIA", crear_dim_geografia)
    ejecutar_con_manejo_errores("DIM_CANAL", crear_dim_canal)

    # ============================================================
    # HECHOS
    # ============================================================
    ejecutar_con_manejo_errores("FACT_TRANSACCIONES", crear_fact_transacciones)
    ejecutar_con_manejo_errores("FACT_CARTERA", crear_fact_cartera)
    ejecutar_con_manejo_errores("FACT_RENTABILIDAD", crear_fact_rentabilidad)

    # ============================================================
    # KPIs
    # ============================================================
    ejecutar_con_manejo_errores("KPI_DIARIO_CARTERA", crear_kpi_diario_cartera)

    # ============================================================
    # LINAJE
    # ============================================================
    ejecutar_con_manejo_errores("LINAJE_DATOS", documentar_linaje)

    # ============================================================
    # GUARDAR ERRORES
    # ============================================================
    guardar_errores_pipeline()

    duration = time.time() - t0

    logger.info(f"\n{'=' * 60}")
    logger.info(f"🥇 GOLD COMPLETADO")
    logger.info(f"{'=' * 60}")
    logger.info(f"""
    Tablas creadas:
      📊 dim_clientes
      📊 dim_productos
      📊 dim_geografia
      📊 dim_canal
      📊 fact_transacciones
      📊 fact_cartera
      📊 fact_rentabilidad_cliente
      📊 kpi_diario_cartera
      📋 gold_linaje_datos
    """)
    logger.info(f"  ⏱️  {duration:.2f}s")
    logger.info("✅ Pipeline Gold completado")

StatementMeta(, 575c5f61-7051-4b2e-8c76-6e33f2202ba1, 36, Finished, Available, Finished, False)

2026-04-07 16:33:20,440 - INFO - 
2026-04-07 16:33:20,441 - INFO - 🥇 PIPELINE GOLD — MODELO ANALÍTICO
2026-04-07 16:33:20,442 - INFO -    Origen:  LH_SILVER_PRUEBA
2026-04-07 16:33:20,442 - INFO -    Destino: LH_GOLD_PRUEBA
2026-04-07 16:33:20,443 - INFO - ============================================================
2026-04-07 16:33:20,443 - INFO - 
2026-04-07 16:33:20,444 - INFO - 📋 dim_clientes
2026-04-07 16:33:20,444 - INFO - ============================================================
2026-04-07 16:33:28,164 - INFO -   ✅ 10,000 registros → `Prueba Tecnica Dataknow`.LH_GOLD_PRUEBA.dbo.dim_clientes
2026-04-07 16:33:28,165 - INFO - ✅ DIM_CLIENTES
2026-04-07 16:33:28,167 - INFO - 
2026-04-07 16:33:28,168 - INFO - 📋 dim_productos
2026-04-07 16:33:28,168 - INFO - ============================================================
2026-04-07 16:33:36,205 - INFO -   ✅ 50 registros → `Prueba Tecnica Dataknow`.LH_GOLD_PRUEBA.dbo.dim_productos
2026-04-07 16:33:36,206 - INFO - ✅ DIM_PRODUCTOS
2026-04